# Demo A · Where the Memory and the Milliseconds Go

**ODSC 2026** | Serving LLMs Efficiently at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)

Seven questions, answered by measurement on whatever machine is running this cell:

| § | Question | Answer we measure |
|---|----------|-------------------|
| 01 | What did we just load? | Shape of the model, from `config.json` |
| 02 | Where did the memory go? | Weights are fixed; the rest is your serving budget |
| 03 | How long before the user sees anything? | TTFT = one forward pass over the prompt |
| 04 | How long between tokens after that? | ITL, one pass per token, KV cache on |
| 05 | Does a longer prompt make generation slower? | TTFT rises, ITL stays flat |
| 06 | How does the KV cache grow? | Linear in context **and** in users — both measured |
| 07 | So how many users fit? | Where each growth curve crosses your budget |

Pure PyTorch + Hugging Face. No vLLM, no external services, nothing simulated.

**Hardware is detected, not assumed.** A CUDA GPU with ≥ 18 GB runs Mistral-7B; anything
smaller — including CPU-only Molab — falls back to a 1.1B model with smaller knobs. The
absolute numbers change with the machine; every shape in the notebook does not.

**A establishes the bottleneck; [Demo B](odsc_2026_demo_b_attention_and_quantization.ipynb)
attacks it.** §07 here ends on a number that does not fit; B is how you make it fit.

## 00 · Setup

Installs, hardware detection, and the knobs every later section reads. Mistral-7B is a
gated repo, so if we are taking the 7B path this cell also checks your Hugging Face login
and says so now rather than failing part-way into a download.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers>=4.56', 'torch', 'accelerate',
                       'matplotlib', 'pandas', 'huggingface_hub'])

import gc, statistics, time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from matplotlib.colors import ListedColormap
from transformers import AutoModelForCausalLM, AutoTokenizer

GIB = 1024 ** 3

# Presenter-readable defaults: big text, no chartjunk.
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12, 'axes.titlesize': 14,
                     'axes.titleweight': 'bold', 'axes.spines.top': False,
                     'axes.spines.right': False})
COOL, WARM, GREY = '#2563eb', '#991b1b', '#cbd5e1'

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / GIB if HAS_CUDA else 0.0
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'CPU'

# One switch decides the model and every knob below it.
USE_7B = HAS_CUDA and VRAM_GIB >= 18
MODEL_ID = 'mistralai/Mistral-7B-v0.1' if USE_7B else 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32
ELEM_BYTES = torch.finfo(DTYPE).bits // 8

print(f'Device    : {GPU_NAME}' + (f'  ({VRAM_GIB:.1f} GiB)' if HAS_CUDA else ''))
print(f'Model     : {MODEL_ID}')
print(f'Precision : {str(DTYPE).replace("torch.", "")}  ({ELEM_BYTES} bytes per value)')
if not USE_7B:
    print('            ^ fallback path: same shapes, smaller numbers.')

In [ ]:
# Knobs. Scaled so that no cell takes more than ~30 s on its device class.
if USE_7B:
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 500, 64, 16
    LENGTHS = [128, 512, 2048, 8192]
    CTX_MEASURED = [128, 512, 1024, 2048, 4096, 8192]
elif HAS_CUDA:
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 500, 64, 16
    LENGTHS = [128, 512, 1024, 2048]
    CTX_MEASURED = [128, 256, 512, 1024, 2048]
else:
    PROMPT_TOKENS, DECODE_STEPS, UNCACHED_STEPS = 128, 16, 8
    LENGTHS = [64, 128, 256, 512]
    CTX_MEASURED = [64, 128, 256, 512]

# §07 is arithmetic, so these stay fixed on every device and stay comparable.
USERS = [1, 8, 32, 80]
CONTEXTS = [4096, 16384]

print(f'Prompt for 03  : {PROMPT_TOKENS} tokens')
print(f'Decode steps   : {DECODE_STEPS} cached, {UNCACHED_STEPS} uncached')
print(f'Length sweep   : {LENGTHS}')
print(f'Capacity grid  : {USERS} users x {[c // 1024 for c in CONTEXTS]}K context')

if USE_7B:
    try:
        from huggingface_hub import whoami
        print(f'HF token       : ok, logged in as {whoami()["name"]}')
    except Exception:
        print('HF token       : NOT FOUND. Run  huggingface-cli login  (or set HF_TOKEN),')
        print(f'                 then accept the licence at https://huggingface.co/{MODEL_ID}')
else:
    print('HF token       : not needed on the fallback path')

In [ ]:
def sync():
    if HAS_CUDA:
        torch.cuda.synchronize()


def timed(fn):
    """Wall time of fn() in seconds, device-synchronized on both sides."""
    sync()
    t0 = time.perf_counter()
    out = fn()
    sync()
    return out, time.perf_counter() - t0


def free_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()


def kv_pairs(past):
    """(key, value) tensors per layer. transformers has shipped several cache
    classes, so normalize once here instead of in every cell below."""
    if hasattr(past, 'layers'):
        return [(lay.keys, lay.values) for lay in past.layers]
    if hasattr(past, 'key_cache'):
        return list(zip(past.key_cache, past.value_cache))
    return [(k, v) for k, v in past]


def kv_bytes(past):
    return sum(t.numel() * t.element_size() for pair in kv_pairs(past) for t in pair)


def kv_tokens(past):
    return int(kv_pairs(past)[0][0].shape[-2])


print('Helpers ready: timed, free_memory, kv_bytes, kv_tokens.')

## 01 · Model Loading

Six numbers from `config.json` decide everything that follows. The one to watch is the
**KV head count against the query head count** — where those differ, that is
grouped-query attention, and it is the only reason the arithmetic in §06 is survivable.

In [ ]:
print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model, load_sec = timed(lambda: AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map='auto' if HAS_CUDA else None).eval())
if not HAS_CUDA:
    model.to(DEVICE)

cfg = model.config
N_LAYERS = cfg.num_hidden_layers
N_HEADS = cfg.num_attention_heads
N_KV_HEADS = getattr(cfg, 'num_key_value_heads', None) or N_HEADS
HEAD_DIM = getattr(cfg, 'head_dim', None) or cfg.hidden_size // N_HEADS
GROUP = N_HEADS // N_KV_HEADS
N_PARAMS = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = sum(p.numel() * p.element_size() for p in model.parameters())

print(f'Load time      : {load_sec:.1f} s')
print(f'Layers         : {N_LAYERS}')
print(f'Hidden size    : {cfg.hidden_size}')
print(f'Query heads    : {N_HEADS}')
print(f'KV heads       : {N_KV_HEADS}' +
      (f'      <- GQA: {GROUP} query heads per KV head' if GROUP > 1 else '      (no GQA)'))
print(f'Head dim       : {HEAD_DIM}')
print(f'Parameters     : {N_PARAMS / 1e9:.2f} B')
print(f'Weights        : {WEIGHT_BYTES / GIB:.2f} GiB at {ELEM_BYTES} bytes each')

In [ ]:
# One square per head. Colour = which KV head it reads from.
group_colors = plt.cm.tab20(range(N_KV_HEADS))

fig, ax = plt.subplots(figsize=(11, 3.2))
for h in range(N_HEADS):
    ax.add_patch(plt.Rectangle((h, 1.2), 0.86, 0.8, color=group_colors[h // GROUP],
                               ec='#000', lw=0.8))
for g in range(N_KV_HEADS):
    ax.add_patch(plt.Rectangle((g * GROUP, 0), GROUP - 0.14, 0.8,
                               color=group_colors[g], ec='#000', lw=1.4))

ax.text(-0.6, 1.6, f'{N_HEADS} query heads', ha='right', va='center', fontweight='bold')
ax.text(-0.6, 0.4, f'{N_KV_HEADS} KV heads', ha='right', va='center', fontweight='bold')
ax.set_xlim(-N_HEADS * 0.3, N_HEADS)
ax.set_ylim(-0.4, 2.4)
ax.axis('off')
ax.set_title(f'{GROUP} query heads per cached KV head\n'
             f'{N_LAYERS} layers x {N_KV_HEADS} KV heads x {HEAD_DIM} dim is what we store',
             fontsize=13)
plt.tight_layout()
plt.show()

## 02 · Memory Distribution

Parameters times bytes-per-parameter. The weights are a **fixed** cost, paid once, shared
by every user. Everything left over is the budget you serve users out of — and on a GPU
we can read that remainder straight off the device.

In [ ]:
sync()
if HAS_CUDA:
    FREE_B, TOTAL_B = torch.cuda.mem_get_info()
    ALLOC_B = torch.cuda.memory_allocated()
    BUDGET_B, BUDGET_LABEL = FREE_B, 'measured free VRAM'
else:
    TOTAL_B = ALLOC_B = WEIGHT_BYTES
    FREE_B = 0
    # No VRAM to measure, so §06-07 need a STATED budget. Labelled as an
    # assumption everywhere it is used, never as a measurement.
    BUDGET_B = 80 * GIB - WEIGHT_BYTES
    BUDGET_LABEL = 'ASSUMED: 80 GiB GPU minus these weights'

BUDGET_GIB = BUDGET_B / GIB

print(f'Weights               : {WEIGHT_BYTES / GIB:6.2f} GiB   ({WEIGHT_BYTES / 1e9:.1f} GB)')
if HAS_CUDA:
    print(f'Allocated on device   : {ALLOC_B / GIB:6.2f} GiB')
    print(f'Total device memory   : {TOTAL_B / GIB:6.2f} GiB')
print(f'Serving budget        : {BUDGET_GIB:6.2f} GiB   <- {BUDGET_LABEL}')
print()

WIDTH = 50
ref_b = TOTAL_B if HAS_CUDA else BUDGET_B + WEIGHT_BYTES
used_frac = (ref_b - BUDGET_B) / ref_b
filled = max(round(WIDTH * used_frac), 1)
print(f'  |{"W" * filled}{"." * (WIDTH - filled)}|  {ref_b / GIB:.1f} GiB')
print(f'   W = weights            {(ref_b - BUDGET_B) / GIB:6.2f} GiB  ({used_frac * 100:.0f}%)')
print(f'   . = free for KV cache  {BUDGET_GIB:6.2f} GiB  ({(1 - used_frac) * 100:.0f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
used_gib = (ref_b - BUDGET_B) / GIB

ax.barh([0], [used_gib], color=COOL, ec='#000', label='weights + runtime')
ax.barh([0], [BUDGET_GIB], left=[used_gib], color=GREY, ec='#000',
        label='free for KV cache')
ax.text(used_gib / 2, 0, f'{used_gib:.1f} GiB', ha='center', va='center',
        color='white', fontweight='bold', fontsize=15)
ax.text(used_gib + BUDGET_GIB / 2, 0, f'{BUDGET_GIB:.0f} GiB', ha='center', va='center',
        fontweight='bold', fontsize=15)

ax.set_yticks([])
ax.set_xlim(0, ref_b / GIB)
ax.set_xlabel('memory (GiB)')
ax.set_title(f'{GPU_NAME}: fixed cost vs serving budget, zero users connected')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.75), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

## 03 · Prefill

**Prefill** is one forward pass over the whole prompt. Every token is processed in
parallel, and the first output token falls out of the end of it — so prefill *is*
time-to-first-token. Warm up once (the first call compiles kernels), then time one pass.

In [ ]:
ESSAY = ('The transformer architecture processes every token of the prompt in parallel '
         'during the forward pass, which makes prompt ingestion fast but generation '
         'strictly sequential. Each generated token must wait for the one before it. ')


def make_prompt(n_tokens):
    """A prompt of exactly n_tokens, built by repeating ESSAY."""
    text = ESSAY * (n_tokens // 20 + 2)
    ids = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=n_tokens)['input_ids']
    return ids.to(DEVICE)


@torch.no_grad()
def prefill(ids, use_cache=True):
    return timed(lambda: model(input_ids=ids, use_cache=use_cache))


PROMPT = make_prompt(PROMPT_TOKENS)
N_PROMPT = PROMPT.shape[1]

prefill(PROMPT)                      # warm-up, not timed
out, prefill_sec = prefill(PROMPT)
TTFT_MS = prefill_sec * 1000
PREFILL_TPS = N_PROMPT / prefill_sec

print(f'Prompt            : {N_PROMPT} tokens')
print(f'TTFT              : {TTFT_MS:.0f} ms      <- one forward pass')
print(f'Per input token   : {TTFT_MS / N_PROMPT:.3f} ms')
print(f'Prefill throughput: {PREFILL_TPS:,.0f} tokens/s')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.8))

ax.barh([0], [TTFT_MS], color=COOL, ec='#000', height=0.5)
ax.text(TTFT_MS / 2, 0, f'prefill: {N_PROMPT} tokens in ONE pass',
        ha='center', va='center', color='white', fontweight='bold', fontsize=13)
ax.annotate(f'first token\n{TTFT_MS:.0f} ms', xy=(TTFT_MS, 0), xytext=(TTFT_MS * 1.08, 0),
            va='center', fontsize=13, fontweight='bold', color=WARM,
            arrowprops=dict(arrowstyle='->', color=WARM, lw=1.6))
ax.text(0, -0.45, 'user sees nothing during this window', fontsize=11, color='#475569')

ax.set_yticks([])
ax.set_xlim(0, TTFT_MS * 1.45)
ax.set_xlabel('time (ms)')
ax.set_title(f'Time to first token at {PREFILL_TPS:,.0f} input tokens/s')
plt.tight_layout()
plt.show()

## 04 · Decode With the KV Cache

After prefill, every further token is its own forward pass with **one** input token —
the earlier keys and values come from `past_key_values` instead of being recomputed.
The gap between tokens is inter-token latency (ITL).

In [ ]:
@torch.no_grad()
def decode_cached(prompt_ids, steps):
    """Generate `steps` tokens one at a time, reusing the KV cache.
    Returns per-step latency in ms."""
    out, _ = prefill(prompt_ids)
    past = out.past_key_values
    tok = out.logits[:, -1:].argmax(-1)

    per_step = []
    for _ in range(steps):
        out, sec = timed(lambda: model(input_ids=tok, past_key_values=past,
                                       use_cache=True))
        past, tok = out.past_key_values, out.logits[:, -1:].argmax(-1)
        per_step.append(sec * 1000)
    return per_step, past


itl_cached, FINAL_CACHE = decode_cached(PROMPT, DECODE_STEPS)

ITL_MEAN = statistics.mean(itl_cached)
ITL_P50 = statistics.median(itl_cached)

print(f'Decoded           : {DECODE_STEPS} tokens, one forward pass each')
print(f'ITL mean          : {ITL_MEAN:.1f} ms')
print(f'ITL p50           : {ITL_P50:.1f} ms')
print(f'Decode throughput : {1000 / ITL_MEAN:.1f} tokens/s  (single stream)')
print(f'Prefill was {PREFILL_TPS / (1000 / ITL_MEAN):,.0f}x more productive per second.')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))

ax.plot(range(1, DECODE_STEPS + 1), itl_cached, 'o-', color=COOL, ms=4, lw=1.6,
        label='measured per step')
ax.axhline(ITL_P50, color=WARM, ls='--', lw=2, label=f'p50 = {ITL_P50:.1f} ms')
ax.axhline(ITL_MEAN, color='#166534', ls=':', lw=2, label=f'mean = {ITL_MEAN:.1f} ms')
ax.set_ylim(0, max(itl_cached) * 1.3)
ax.set_xlabel('decode step')
ax.set_ylabel('inter-token latency (ms)')
ax.set_title(f'{DECODE_STEPS} tokens with the KV cache: flat at '
             f'{ITL_P50:.0f} ms/token, {1000 / ITL_MEAN:.0f} tok/s')
ax.legend()
plt.tight_layout()
plt.show()

### 04b · The same tokens with `use_cache=False`

Optional, and worth the seconds: without the cache, every step re-forwards the whole
sequence, so the cost per token climbs as the sequence grows.

In [ ]:
@torch.no_grad()
def decode_uncached(prompt_ids, steps):
    """Same output, no cache: re-forward the full sequence every step."""
    out, _ = prefill(prompt_ids, use_cache=False)
    seq = torch.cat([prompt_ids, out.logits[:, -1:].argmax(-1)], dim=1)

    per_step = []
    for _ in range(steps):
        out, sec = timed(lambda: model(input_ids=seq, use_cache=False))
        seq = torch.cat([seq, out.logits[:, -1:].argmax(-1)], dim=1)
        per_step.append(sec * 1000)
    return per_step


free_memory()
itl_uncached = decode_uncached(PROMPT, UNCACHED_STEPS)

print(f'With cache   : {ITL_MEAN:6.1f} ms/token mean, flat')
print(f'Without cache: {statistics.mean(itl_uncached):6.1f} ms/token mean, '
      f'{itl_uncached[0]:.0f} -> {itl_uncached[-1]:.0f} ms across {UNCACHED_STEPS} steps')
print(f'Cache is worth {statistics.mean(itl_uncached) / ITL_MEAN:.0f}x at {N_PROMPT} tokens '
      f'of context, and more at every longer one.')

In [ ]:
n_un = len(itl_uncached)
mean_on = statistics.mean(itl_cached[:n_un])
mean_off = statistics.mean(itl_uncached)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))

ax1.plot(range(1, n_un + 1), itl_cached[:n_un], 'o-', color=COOL, ms=5, lw=1.8,
         label='use_cache=True')
ax1.plot(range(1, n_un + 1), itl_uncached, 'o-', color=WARM, ms=5, lw=1.8,
         label='use_cache=False')
ax1.set_ylim(0, mean_off * 1.6)
ax1.set_xlabel('decode step')
ax1.set_ylabel('ITL (ms)')
ax1.set_title(f'Same {n_un} tokens, cache on vs off')
ax1.legend()

ax2.bar(['with cache', 'no cache'], [mean_on, mean_off],
        color=[COOL, WARM], ec='#000', width=0.55)
for i, v in enumerate([mean_on, mean_off]):
    ax2.text(i, v, f'{v:.0f} ms', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax2.set_ylim(0, mean_off * 1.3)
ax2.set_ylabel('mean ms per token')
ax2.set_title(f'{mean_off / mean_on:.0f}x, and only {N_PROMPT} tokens of context')

plt.tight_layout()
plt.show()

## 05 · Short vs Long Prompts

Four prompt lengths. For each: TTFT from one prefill, and mean ITL over 32 decoded
tokens. The two columns behave completely differently — which is the single most useful
fact in this notebook.

In [ ]:
rows = []

for n in LENGTHS:
    ids = make_prompt(n)
    prefill(ids)                                  # warm-up at this shape
    _, sec = prefill(ids)
    steps, _ = decode_cached(ids, 32)
    rows.append({'Prompt tokens': int(ids.shape[1]),
                 'TTFT (ms)': round(sec * 1000, 1),
                 'ms per input token': round(sec * 1000 / ids.shape[1], 3),
                 'ITL mean (ms)': round(statistics.mean(steps), 1),
                 'Decode tok/s': round(1000 / statistics.mean(steps), 1)})
    free_memory()

length_table = pd.DataFrame(rows)
display(length_table)

ttft_growth = length_table['TTFT (ms)'].iloc[-1] / length_table['TTFT (ms)'].iloc[0]
itl_growth = length_table['ITL mean (ms)'].iloc[-1] / length_table['ITL mean (ms)'].iloc[0]
print(f'{LENGTHS[0]} -> {LENGTHS[-1]} tokens is {LENGTHS[-1] // LENGTHS[0]}x more prompt.')
print(f'  TTFT grew {ttft_growth:.0f}x')
print(f'  ITL  grew {itl_growth:.2f}x   <- essentially flat')

In [ ]:
xs = length_table['Prompt tokens']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))

ax1.plot(xs, length_table['TTFT (ms)'], 'o-', color=COOL, ms=7, lw=2)
for x, y in zip(xs, length_table['TTFT (ms)']):
    ax1.annotate(f'{y:.0f} ms', (x, y), textcoords='offset points', xytext=(0, 9),
                 ha='center', fontsize=11)
ax1.set_ylim(0, length_table['TTFT (ms)'].max() * 1.3)
ax1.set_xlabel('prompt tokens')
ax1.set_ylabel('TTFT (ms)')
ax1.set_title('TTFT scales with the prompt')

ax2.plot(xs, length_table['ITL mean (ms)'], 'o-', color=WARM, ms=7, lw=2)
for x, y in zip(xs, length_table['ITL mean (ms)']):
    ax2.annotate(f'{y:.1f} ms', (x, y), textcoords='offset points', xytext=(0, 9),
                 ha='center', fontsize=11)
ax2.set_ylim(0, length_table['ITL mean (ms)'].max() * 1.8)
ax2.set_xlabel('prompt tokens')
ax2.set_ylabel('ITL (ms)')
ax2.set_title('ITL barely notices')

for ax in (ax1, ax2):
    ax.set_xscale('log', base=2)
    ax.set_xticks(LENGTHS)
    ax.set_xticklabels([str(n) for n in LENGTHS])

plt.tight_layout()
plt.show()

## 06 · How the KV Cache Grows

$$\text{KV bytes per token} = 2 \times L \times n_{kv} \times d_h \times \text{bytes per value}$$

Leading 2 is one K plus one V. Note $n_{kv}$ is the **KV** head count, not the query head
count — that substitution is what GQA buys. Predict it from the config, then weigh the
real cache from §04 and check.

That is a cost *per token*, and it is paid **per token, per user**. So the cache grows in
two independent directions at once:

$$\text{KV cache} = \underbrace{\text{bytes per token}}_{\text{fixed by architecture}}
\times \underbrace{\text{context length}}_{\text{06a}}
\times \underbrace{\text{concurrent users}}_{\text{06b}}$$

The weights were one number you pay once. This is a product of three, and you control two.

In [ ]:
KV_PER_TOKEN = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * ELEM_BYTES

print('PREDICTED from config.json')
print(f'  2 x {N_LAYERS} layers x {N_KV_HEADS} kv-heads x {HEAD_DIM} dim x {ELEM_BYTES} B'
      f'  =  {KV_PER_TOKEN:,} bytes  ({KV_PER_TOKEN / 1024:.0f} KiB) per token')
print()

measured_tokens = kv_tokens(FINAL_CACHE)
measured_per_token = kv_bytes(FINAL_CACHE) / measured_tokens
print('MEASURED from the real past_key_values in §04')
print(f'  {kv_bytes(FINAL_CACHE) / 1e6:,.1f} MB over {measured_tokens} tokens'
      f'  =  {measured_per_token:,.0f} bytes per token')
print(f'  error vs prediction: '
      f'{abs(measured_per_token - KV_PER_TOKEN) / KV_PER_TOKEN * 100:.2f}%')
print()
print('So the architecture fixes the per-token price. The next two cells are about')
print('the two things that multiply it.')

### 06a · Direction one: context length

Prefill at several lengths and **weigh the real cache each time** — no arithmetic. If the
formula is right, the measured points land on a straight line whose slope is exactly the
per-token cost above.

In [ ]:
growth = []

for n in CTX_MEASURED:
    ids = make_prompt(n)
    out, _ = prefill(ids)
    growth.append((int(ids.shape[1]), kv_bytes(out.past_key_values)))
    del out
    free_memory()

growth_table = pd.DataFrame([
    {'Context tokens': n,
     'Cache measured (MiB)': round(b / 1024 ** 2, 1),
     'Cache predicted (MiB)': round(KV_PER_TOKEN * n / 1024 ** 2, 1),
     'Bytes per token': round(b / n)}
    for n, b in growth])
display(growth_table)

slope = (growth[-1][1] - growth[0][1]) / (growth[-1][0] - growth[0][0])
print(f'Measured slope : {slope:,.0f} bytes/token ({slope / 1024:.0f} KiB) '
      f'vs predicted {KV_PER_TOKEN / 1024:.0f} KiB')
print('Double the context, double the cache. Exactly linear — no compression,')
print('no decay, nothing that gets cheaper at scale.')

In [ ]:
xs = [n for n, _ in growth]
measured_gib = [b / GIB for _, b in growth]

fig, ax = plt.subplots(figsize=(11, 4.4))

ax.plot(xs, [KV_PER_TOKEN * n / GIB for n in xs], '-', color=GREY, lw=7,
        label=f'predicted: {KV_PER_TOKEN / 1024:.0f} KiB x context')
ax.plot(xs, measured_gib, 'o-', color=COOL, ms=8, lw=2, label='measured cache')
for n, g in zip(xs, measured_gib):
    ax.annotate(f'{g * 1024:.0f} MiB', (n, g), textcoords='offset points',
                xytext=(0, 11), ha='center', fontsize=10)

ax.set_xlabel('context length (tokens)')
ax.set_ylabel('KV cache, ONE user (GiB)')
ax.set_ylim(0, max(measured_gib) * 1.35)
ax.set_title('One user: the cache is exactly linear in context length')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 06b · Direction two: concurrent users

Every user carries their own cache, so the second direction is a plain multiplier. Sweep
context on the x-axis, draw one line per user count, and put the §02 budget across it as a
horizontal line. The question stops being abstract: not *how big is the cache* but
**where does each line cross the budget**.

In [ ]:
CTX_SWEEP = [512, 2048, 4096, 8192, 16384, 32768]
MODEL_MAX = getattr(cfg, 'max_position_embeddings', None)

users_table = pd.DataFrame(
    [[f'{KV_PER_TOKEN * u * c / GIB:,.1f}' for c in CTX_SWEEP] for u in USERS],
    index=[f'{u} users' for u in USERS],
    columns=[f'{c // 1024}K' if c >= 1024 else str(c) for c in CTX_SWEEP])
print('KV cache in GiB, users (down) x context (across):')
display(users_table)

print(f'Serving budget: {BUDGET_GIB:.1f} GiB  ({BUDGET_LABEL})')
print()
print('Longest context each user count can afford:')
CROSSINGS = {}
for u in USERS:
    max_ctx = BUDGET_B / (KV_PER_TOKEN * u)
    CROSSINGS[u] = max_ctx
    note = ''
    if MODEL_MAX and max_ctx > MODEL_MAX:
        note = f'   (model caps you at {MODEL_MAX:,} first)'
    print(f'  {u:>3} users -> {max_ctx:>9,.0f} tokens{note}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8))

# Left: both directions at once, against the budget.
for i, u in enumerate(USERS):
    ys = [KV_PER_TOKEN * u * c / GIB for c in CTX_SWEEP]
    ax1.plot(CTX_SWEEP, ys, 'o-', ms=5, lw=2,
             color=plt.cm.viridis(i / max(len(USERS) - 1, 1)), label=f'{u} users')
    x_cross = CROSSINGS[u]
    if CTX_SWEEP[0] <= x_cross <= CTX_SWEEP[-1]:
        ax1.plot([x_cross], [BUDGET_GIB], 'X', ms=14, color=WARM, zorder=5)
        ax1.annotate(f'{x_cross / 1024:.1f}K', (x_cross, BUDGET_GIB),
                     textcoords='offset points', xytext=(6, 10), fontsize=11,
                     fontweight='bold', color=WARM)

ax1.axhline(BUDGET_GIB, color='#166534', ls='--', lw=2.5,
            label=f'budget = {BUDGET_GIB:.0f} GiB')
if MODEL_MAX and CTX_SWEEP[0] <= MODEL_MAX <= CTX_SWEEP[-1]:
    ax1.axvline(MODEL_MAX, color='#475569', ls=':', lw=1.8)
    ax1.text(MODEL_MAX, BUDGET_GIB * 2.05, ' model limit', fontsize=10, color='#475569')

ax1.set_xscale('log', base=2)
ax1.set_xticks(CTX_SWEEP)
ax1.set_xticklabels([f'{c // 1024}K' if c >= 1024 else str(c) for c in CTX_SWEEP])
ax1.set_ylim(0, BUDGET_GIB * 2.2)
ax1.set_xlabel('context length')
ax1.set_ylabel('KV cache (GiB)')
ax1.set_title('X marks where you run out of memory')
ax1.legend(fontsize=9, loc='upper left')

# Right: the same thing as a fits/OOM grid at the two contexts that matter.
grid = [[KV_PER_TOKEN * u * c / GIB for c in CONTEXTS] for u in USERS]
fits = [[0 if v <= BUDGET_GIB else 1 for v in row] for row in grid]
ax2.imshow(fits, cmap=ListedColormap(['#dcfce7', '#fecaca']), vmin=0, vmax=1,
           aspect='auto')
for i, row in enumerate(grid):
    for j, v in enumerate(row):
        tag = '' if v <= BUDGET_GIB else '\nOOM'
        ax2.text(j, i, f'{v:,.1f} GiB{tag}', ha='center', va='center',
                 fontsize=12, fontweight='bold',
                 color='#166534' if v <= BUDGET_GIB else '#991b1b')
ax2.set_xticks(range(len(CONTEXTS)), [f'{c // 1024}K context' for c in CONTEXTS])
ax2.set_yticks(range(len(USERS)), [f'{u} users' for u in USERS])
ax2.set_title(f'Against {BUDGET_GIB:.0f} GiB of budget')

plt.tight_layout()
plt.show()

## 07 · So How Many Users Fit?

Read it straight off the crossings: budget divided by what one user costs at that context.

On the fallback path the second cell restates the same arithmetic for Mistral-7B on an
80 GiB GPU — **derived from published config, not measured here**, and labelled as such in
the output. It is the version the slides quote.

In [ ]:
cap_rows = []
for c in CONTEXTS:
    per_user_gib = KV_PER_TOKEN * c / GIB
    cap_rows.append({'Context': f'{c // 1024}K tokens',
                     'KV per user (GiB)': round(per_user_gib, 3),
                     'Users that fit': int(BUDGET_GIB / per_user_gib),
                     'At 80 users (GiB)': round(per_user_gib * 80, 1),
                     '80 users?': 'fits' if per_user_gib * 80 <= BUDGET_GIB else 'OOM'})

MAX_USERS = pd.DataFrame(cap_rows)
display(MAX_USERS)

worst = max(CONTEXTS)
need_gib = KV_PER_TOKEN * worst * 80 / GIB
print(f'Budget: {BUDGET_GIB:.1f} GiB  ({BUDGET_LABEL})')
print()
print(f'80 users at {worst // 1024}K context needs {need_gib:,.0f} GiB '
      f'= {need_gib / BUDGET_GIB:.1f}x what we have.')
print('Nothing about that is fixable by waiting for a faster GPU. The only moves are')
print('to shrink a term in the product — which is Demo B.')

In [ ]:
# If we are on the fallback path, the numbers above are real but small. The
# production-shaped version of the same arithmetic is worth seeing either way,
# so compute it from Mistral-7B's PUBLISHED config and label it as derived.
REF_NAME = 'Mistral-7B-v0.1 @ bf16 on an 80 GiB GPU'
REF_PER_TOKEN = 2 * 32 * 8 * 128 * 2          # layers, kv-heads, head_dim, bytes
REF_WEIGHTS_GIB = 13.5
REF_BUDGET_GIB = 80.0 - REF_WEIGHTS_GIB

if USE_7B:
    print('Measured above IS the reference configuration — nothing to add.')
else:
    here = MODEL_ID.split('/')[-1]
    print(f'DERIVED, not measured here: {REF_NAME}')
    print(f'  KV per token   : {REF_PER_TOKEN / 1024:.0f} KiB'
          f'   (vs {KV_PER_TOKEN / 1024:.0f} KiB measured on {here})')
    print(f'  Serving budget : {REF_BUDGET_GIB:.1f} GiB')
    for u, c in [(80, 4096), (80, 16384)]:
        gib = REF_PER_TOKEN * u * c / GIB
        verdict = 'fits' if gib <= REF_BUDGET_GIB else f'OOM, {gib / REF_BUDGET_GIB:.1f}x the GPU'
        print(f'  {u} users x {c // 1024}K = {gib:5.0f} GiB   -> {verdict}')
    print()
    print('Same shape, bigger numbers. This is the arithmetic the slides quote, and')
    print('it comes from config.json — which is the whole point of 06.')

## 08 · Summary — Every Number We Measured

One machine, no simulation. The only predicted value in the notebook (KV bytes per token)
was weighed against the real cache twice: once in §06 and once as a slope in §06a.

In [ ]:
# 04b is optional, so fall back cleanly if the room skipped it.
no_cache = (f'{statistics.mean(itl_uncached):.1f} ms'
            if 'itl_uncached' in globals() else 'not run')

summary = pd.DataFrame([
    ('01  Model',    'Parameters',                   f'{N_PARAMS / 1e9:.2f} B',
     f'{MODEL_ID.split("/")[-1]} at {ELEM_BYTES} bytes/value'),
    ('01  Model',    'Layers x KV heads x head dim', f'{N_LAYERS} x {N_KV_HEADS} x {HEAD_DIM}',
     f'GQA: {GROUP} query heads per KV head'),
    ('02  Memory',   'Weights',                      f'{WEIGHT_BYTES / GIB:.2f} GiB',
     'fixed cost, shared by all users'),
    ('02  Memory',   'Serving budget',               f'{BUDGET_GIB:.1f} GiB',
     BUDGET_LABEL),
    ('03  Prefill',  f'TTFT, {N_PROMPT}-token prompt', f'{TTFT_MS:.0f} ms',
     f'{PREFILL_TPS:,.0f} input tok/s, one pass'),
    ('04  Decode',   'ITL mean / p50',               f'{ITL_MEAN:.1f} / {ITL_P50:.1f} ms',
     f'{1000 / ITL_MEAN:.0f} tok/s, single stream'),
    ('04  Decode',   'ITL without KV cache',         no_cache,
     'and climbing with every step'),
    ('05  Length',   f'TTFT, {LENGTHS[0]} -> {LENGTHS[-1]} tokens', f'{ttft_growth:.0f}x',
     'prefill is length-proportional'),
    ('05  Length',   f'ITL, {LENGTHS[0]} -> {LENGTHS[-1]} tokens',  f'{itl_growth:.2f}x',
     'decode is length-indifferent'),
    ('06  Growth',   'KV cache per token',           f'{KV_PER_TOKEN / 1024:.0f} KiB',
     f'predicted and measured, {abs(measured_per_token - KV_PER_TOKEN) / KV_PER_TOKEN * 100:.2f}% apart'),
    ('06a Growth',   'Measured slope vs context',    f'{slope / 1024:.0f} KiB/token',
     'linear, across a 64x range of context'),
    ('06b Growth',   'Longest context at 80 users',  f'{CROSSINGS[80] / 1024:,.1f}K tokens',
     'where the 80-user line crosses the budget'),
    ('07  Capacity', f'Users at {CONTEXTS[0] // 1024}K context',
     f'{int(BUDGET_GIB / (KV_PER_TOKEN * CONTEXTS[0] / GIB))}',
     f'{KV_PER_TOKEN * CONTEXTS[0] / GIB:.2f} GiB each'),
    ('07  Capacity', f'Users at {CONTEXTS[-1] // 1024}K context',
     f'{int(BUDGET_GIB / (KV_PER_TOKEN * CONTEXTS[-1] / GIB))}',
     f'{KV_PER_TOKEN * CONTEXTS[-1] / GIB:.2f} GiB each'),
], columns=['Section', 'Measurement', 'Value', 'What it means'])

display(summary)

### What we discussed, in one table

| What we found | Why it is true | What it forces you to do |
|---|---|---|
| The weights are a fixed, one-time cost | Parameters x bytes-per-parameter, loaded once | Pick the GPU from the weights; plan capacity from what is left |
| The free remainder is the whole serving budget | Nothing else shares it | Every user you add is a claim on one number |
| TTFT is one parallel pass | Prefill processes all prompt tokens at once | Long prompts cost latency *before* the first character |
| ITL is one pass per token, and flat | `past_key_values` means earlier K/V are never recomputed | Decode speed is set by the model and GPU, not the prompt |
| Without the cache, per-token cost climbs | Each step re-forwards the whole sequence | The KV cache is not an optimization, it is the design |
| 64x the prompt moved TTFT a lot and ITL almost none | Prefill is compute-shaped, decode is memory-shaped | Two phases, two bottlenecks — never quote one average |
| KV bytes/token matches the config formula exactly | `2 x L x n_kv x d_h x bytes`, checked against the real cache | Capacity planning is arithmetic on `config.json` |
| **The cache is linear in context** | Measured, across a 64x range — no compression, no decay | Context length is a capacity decision, not a product one |
| **The cache is linear in users too** | Every user carries their own | The two multiply: doubling both quadruples the cache |
| The 80-user line crosses the budget early | Budget / (bytes-per-token x context) | Concurrency and context trade directly against each other |

---

**Every number in this demo, measured. Next: why they make it slow.**

§07 is the handoff: the load we want needs more KV cache than we have. Two terms in that
product are attackable — the KV cache per token (GQA → MQA → MLA) and the weights
(INT8 → INT4). That is [Demo B](odsc_2026_demo_b_attention_and_quantization.ipynb).